In [39]:
!pip install pyspark

In [40]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataLakeAssignment") \
    .getOrCreate()

df = spark.read.csv(
    "/content/data_lake/bronze/sales.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id| customer_name|      product|category|quantity|unit_price|discount_percent|payment_method|      city|         state|order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|  100001|      C0256|Keerthi Mishra|        Saree|Clothing|       2|      6710|              20|   Credit Card|    Mumbai|   Maharashtra|2026-01-01|   Delivered|
|  100002|      C0042|   Arjun Singh|      T-Shirt|Clothing|       6|       680|               5|    Debit Card|   Chennai|    Tamil Nadu|2025-08-03|   Delivered|
|  100003|      C0348|   Keerthi Rao|       Hoodie|Clothing|       6|      3080|              12|           UPI|      Pune|   Maharashtra|2025-05-06|     Shipped|
|  100004|      C0051|

In [41]:

from pyspark.sql import SparkSession

# Initialize Spark
spark = SparkSession.builder \
    .appName("DataLakeAssignment") \
    .getOrCreate()

# Load raw data from Bronze
df = spark.read.csv(
    "data_lake/bronze/sales.csv",
    header=True,
    inferSchema=True
)

print("Spark session created!")


Spark session created!


In [42]:

# Display schema
df.printSchema()

# Display first 5 rows
df.show(5, truncate=False)

# Total number of records
print("Total records:", df.count())

# Column names
print("Columns:", df.columns)


root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------

In [43]:

# Count exact duplicate rows
duplicate_count = df.count() - df.dropDuplicates().count()

print("Duplicate rows:", duplicate_count)


Duplicate rows: 10


In [44]:

from pyspark.sql.functions import col, sum, when

null_counts = df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

null_counts.show()


+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [45]:

from pyspark.sql.functions import trim

# Replace this with the actual name of a text column
text_column = "Product"

if text_column in df.columns:
    df.select(text_column).distinct().show(50, truncate=False)


In [46]:
df.select(
    "quantity",
    "unit_price",
    "discount_percent"
).describe().show()

+-------+------------------+------------------+------------------+
|summary|          quantity|        unit_price|  discount_percent|
+-------+------------------+------------------+------------------+
|  count|              1000|              1000|              1000|
|   mean|             5.355|            5702.4|            19.948|
| stddev|2.9916425128890194|10990.194484849431|14.236782711763993|
|    min|                -2|              -500|                -5|
|    max|                10|             81010|               120|
+-------+------------------+------------------+------------------+



In [47]:
from pyspark.sql.functions import col

df.filter(
    (col("quantity") < 0) |
    (col("unit_price") < 0) |
    (~col("discount_percent").between(0, 100))
).show()

+--------+-----------+----------------+--------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|   customer_name|             product|       category|quantity|unit_price|discount_percent|  payment_method|      city|         state|order_date|order_status|
+--------+-----------+----------------+--------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|  100034|      C0192|    Divya Sharma|            Yoga Mat|         Sports|       7|      1720|              -5|Cash on Delivery|Coimbatore|    Tamil Nadu|2025-08-19|   Delivered|
|  100140|      C0075| Swathi Tripathi|      Cooking Oil 5L|        Grocery|       4|       790|              -5|     Credit Card| Hyderabad|     Telangana|2025-10-01|     Shipped|
|  100198|      C0014|    Arjun Pillai|              Hoodie|       Clothing|      -2|      2160

In [48]:

# Replace with your actual date column
date_column = "Order Date"

if date_column in df.columns:
    df.select(date_column).distinct().show(30, truncate=False)


In [49]:

from pyspark.sql.functions import col, trim, upper

silver_df = df.dropDuplicates()

print("Rows after removing duplicates:", silver_df.count())


Rows after removing duplicates: 990


In [50]:

# Replace with your actual essential column names
silver_df = silver_df.filter(
    col("order_id").isNotNull()
)


In [51]:

from pyspark.sql.functions import initcap

# Replace with the actual text columns
text_columns = ["Product"]

for c in text_columns:
    if c in silver_df.columns:
        silver_df = silver_df.withColumn(
            c,
            initcap(trim(col(c)))
        )

silver_df.show(5, truncate=False)


+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports   |2       |670       |30              |Credit Card     |Bengaluru |Karnataka  |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books    |4       |670       |0               |Net Banking     |Pune      |Maharashtra|2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books    |6       |610       |12              |UPI      

In [53]:

# Use your exact column names
silver_df = silver_df.filter(
    (col("quantity") >= 0) &
    (col("unit_price") >= 0) &
    (col("discount_percent").between(0, 100))
)


In [54]:
from pyspark.sql.functions import col, trim, initcap

# Start with original Bronze data
silver_df = df.dropDuplicates()

# Handle missing essential order IDs
silver_df = silver_df.filter(
    col("order_id").isNotNull()
)

# Clean text columns
text_columns = [
    "customer_name", "product", "category",
    "payment_method", "city", "state", "order_status"
]

for c in text_columns:
    silver_df = silver_df.withColumn(
        c, initcap(trim(col(c)))
    )

# Validate numerical columns
silver_df = silver_df.filter(
    (col("quantity") >= 0) &
    (col("unit_price") >= 0) &
    (col("discount_percent").between(0, 100))
)

In [55]:
from pyspark.sql.functions import expr

# Convert valid dates to yyyy-MM-dd
# Invalid dates become NULL
silver_df = silver_df.withColumn(
    "order_date",
    expr("""
        date_format(
            try_to_timestamp(order_date, 'dd/MM/yyyy'),
            'yyyy-MM-dd'
        )
    """)
)

silver_df.show(10, truncate=False)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |NULL      |Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |NULL      |Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books      

In [56]:
silver_df.printSchema()
silver_df.show(10, truncate=False)

print("Original row count:", df.count())
print("Silver row count:", silver_df.count())

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+--------------

In [57]:
output_path = "/content/data_lake/silver/preprocessed_sales"

silver_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(output_path)

print("Silver data saved successfully!")

Silver data saved successfully!


In [58]:
silver_read = spark.read.csv(
    output_path,
    header=True,
    inferSchema=True
)

print("Saved row count:", silver_read.count())

silver_read.show(5, truncate=False)

Saved row count: 972
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports   |2       |670       |30              |Credit Card     |Bengaluru |Karnataka  |NULL      |Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books    |4       |670       |0               |Net Banking     |Pune      |Maharashtra|NULL      |Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books    |6       |610       |12   